## 1. 에이전트 개요

- **이름**: 지오 마스터 에이전트
- **목적**: 국가 간 지정학적 이슈를 분석하고 타임라인 카툰 스토리를 생성
- **핵심 기능**:
  1. 지정학적 이슈 국가 추출
  2. 타임라인 카툰 텍스트 및 프롬프트 생성
  3. 타임라인 카툰 이미지 생성 및 출력

## 2. 그래프 구조 (Graph Structure)

- **State**: `input_country`, `related_countries`, `timeline_story`
- **Nodes**:
  - `find_issues`: 입력된 국가와 관련된 최근 100년 내 이슈 국가 검색
  - `create_cartoon`: 이슈를 타임라인 카툰 형식의 텍스트로 변환
- **Edges**: `START` -> `find_issues` -> `create_cartoon` -> `END`

In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# .env 파일 로드
load_dotenv() 

# OpenAI API 키 확인
if not os.getenv("OPENAI_API_KEY"):
    print("경고: OPENAI_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요.")

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

In [2]:
# 1. State 정의
class AgentState(TypedDict):
    input_country: str
    related_countries: List[str]
    timeline_story: str

# 2. Nodes 구현
def find_issues(state: AgentState):
    """입력 국가와 관련된 최근 100년 내 지정학적 이슈 국가 검색"""
    country = state["input_country"]
    prompt = f"{country}와 최근 100년 이내에 중요한 지정학적 이슈가 있었던 국가 3곳을 콤마로 구분해서 이름만 알려줘."
    response = llm.invoke([HumanMessage(content=prompt)])
    countries = [c.strip() for c in response.content.split(",")]
    return {"related_countries": countries}

def create_cartoon(state: AgentState):
    """이슈를 타임라인 카툰 형식의 텍스트로 변환"""
    target = state["input_country"]
    partners = ", ".join(state["related_countries"])
    prompt = f"{target}와 {partners} 사이의 주요 지정학적 사건들을 4컷 카툰 형식의 타임라인(이미지 묘사 + 대사)으로 작성해줘."
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"timeline_story": response.content}

# 3. 그래프 연결
workflow = StateGraph(AgentState)

workflow.add_node("find_issues", find_issues)
workflow.add_node("create_cartoon", create_cartoon)

workflow.add_edge(START, "find_issues")
workflow.add_edge("find_issues", "create_cartoon")
workflow.add_edge("create_cartoon", END)

app = workflow.compile()
print("그래프 컴파일 완료!")

그래프 컴파일 완료!


In [3]:
# 테스트 실행: '대한민국' 입력
inputs = {"input_country": "대한민국"}

print("에이전트 실행 중...\n")
config = {"configurable": {"thread_id": "geo_test_001"}}

for event in app.stream(inputs, config):
    for node_name, output in event.items():
        print(f"[{node_name}] 작업 완료")
        if "timeline_story" in output:
            print("\n=== 최종 타임라인 카툰 시나리오 ===\n")
            print(output["timeline_story"])

에이전트 실행 중...

[find_issues] 작업 완료
[create_cartoon] 작업 완료

=== 최종 타임라인 카툰 시나리오 ===

**4컷 카툰 형식의 타임라인: 대한민국과 일본, 중국, 북한 사이의 주요 지정학적 사건들**

---

### 첫 번째 컷: 일본과의 관계 
**이미지 설명:** 한반도의 지도 위에 대한민국과 일본의 지도. 두 나라 사이에는 "상호 협력"이라는 글씨가 써져 있는 다리가 놓여져 있다.

**대사:** 
대한민국: "우리는 경제 협력을 통해 더 강해질 수 있어요!"
일본: "맞아요, 함께 발전합시다!"

---

### 두 번째 컷: 중국과의 무역
**이미지 설명:** 중국과 대한민국을 연결하는 무역선이 그려져 있고, 그 위에 다양한 제품들이 실려 있다.

**대사:**
대한민국: "중국과의 무역이 늘어나고 있어요!"
중국: "우리의 경제 파트너십은 매우 중요합니다!"

---

### 세 번째 컷: 북한의 도발
**이미지 설명:** 북한의 지도 위에 미사일이 발사되는 그림. 대한민국과 일본이 긴장한 표정을 짓고 있다.

**대사:**
대한민국: "북한이 또 도발하고 있어요!"
일본: "우리는 안전을 지켜야 해요!"

---

### 네 번째 컷: 외교적 대응
**이미지 설명:** 대한민국, 일본, 중국의 대표들이 함께 모여 회의하는 모습. 테이블 위에는 "평화 회담"이라는 팻말이 있다.

**대사:**
대한민국: "협력을 통해 평화를 이뤄냅시다!"
중국: "모두의 안정을 위해 함께 노력해요!"

---

이런 형식으로 카툰을 만들 수 있습니다. 각 컷의 이미지는 상황을 잘 전달하면서도, 유머를 통해 더 쉽게 이해할 수 있도록 디자인할 수 있습니다.
